# PEM hydrogen production rate

Goal: estimate how much hydrogen the electrolyzer produces from electrical input.

For each charge-discharge test I use the charge part and calculate:

input_charge_C = integral(I_charge dt)

input_energy_J = integral(V_charge I_charge dt)

hydrogen_per_J = measured_hydrogen_volume_mL / input_energy_J

The volume readings are manual, so I compare all tests and use an aggregate value rather than trusting
one single test.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This makes the notebook work both from the repo root and from its own folder.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find the project root folder containing data/")
    PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)


## 1. Load volume readings and test files


In [ ]:
readings_file = PROJECT_ROOT / "data/PEM_test/volume_readings/readings.csv"
pem_folder = PROJECT_ROOT / "data/PEM_test/charge_discharge"

volume_readings = pd.read_csv(readings_file)
test_files = sorted(pem_folder.glob("discharge_*s_r1.csv"))

display(volume_readings)
print("Test files:", len(test_files))


## 2. Extract the active electrolysis part of each file


In [ ]:
def load_pem_charge_part(file):
    df = pd.read_csv(file)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["time_s"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()
    df["pem_voltage_V"] = df["ina4_bus_V"] - 0.064
    df["pem_current_A"] = 0.843 * (df["ina4_current_mA"] / 1000) + 0.001
    df["pem_charge_power_W"] = df["pem_voltage_V"] * df["pem_current_A"]

    charge = df[(df["scenario"] == 3) & (df["pem_current_A"] > 0.05)].copy()
    charge["charge_time_s"] = charge["time_s"] - charge["time_s"].iloc[0]
    return charge

charge_tests = {file.name: load_pem_charge_part(file) for file in test_files}

first_name = test_files[0].name
display(charge_tests[first_name].head())


## 3. Plot one charge example to check the selected region


In [ ]:
example_name = "discharge_040A_120s_r1.csv"
example = charge_tests[example_name]

plt.figure(figsize=(10, 4))
plt.plot(example["charge_time_s"], example["pem_current_A"], label="charge current")
plt.plot(example["charge_time_s"], example["pem_voltage_V"], label="charge voltage")
plt.title(example_name)
plt.xlabel("Charge time [s]")
plt.grid(True)
plt.legend()
plt.show()


## 4. Integrate charge and energy for every test


In [ ]:
def parse_setpoints(file_name):
    # Example: discharge_040A_120s_r1.csv -> 0.40 A and 120 s.
    parts = file_name.replace(".csv", "").split("_")
    current_A = int(parts[1].replace("A", "")) / 100
    duration_s = int(parts[2].replace("s", ""))
    return current_A, duration_s

rows = []
for file_name, charge in charge_tests.items():
    current_setpoint_A, duration_setpoint_s = parse_setpoints(file_name)
    charge = charge.copy()
    charge["dt_s"] = charge["charge_time_s"].diff().fillna(0)
    input_charge_C = (charge["pem_current_A"] * charge["dt_s"]).sum()
    input_energy_J = (charge["pem_charge_power_W"] * charge["dt_s"]).sum()

    volume_match = volume_readings[
        (volume_readings["current_A"] == current_setpoint_A)
        & (volume_readings["time_s"] == duration_setpoint_s)
    ]
    hydrogen_volume_mL = volume_match["volume_mL"].iloc[0]

    rows.append(
        {
            "file": file_name,
            "current_setpoint_A": current_setpoint_A,
            "duration_setpoint_s": duration_setpoint_s,
            "hydrogen_volume_mL": hydrogen_volume_mL,
            "input_charge_C": input_charge_C,
            "input_energy_J": input_energy_J,
            "hydrogen_mL_per_C": hydrogen_volume_mL / input_charge_C,
            "hydrogen_mL_per_J": hydrogen_volume_mL / input_energy_J,
        }
    )

charge_summary = pd.DataFrame(rows).sort_values(["current_setpoint_A", "duration_setpoint_s"])
display(charge_summary)


## 5. Compare the rate values


In [ ]:
FARADAY_THEORETICAL_H2_ML_PER_C = 0.12678

SHORT_TEST_HYDROGEN_PRODUCTION_ML_PER_INPUT_J = (
    charge_summary["hydrogen_volume_mL"].sum() / charge_summary["input_energy_J"].sum()
)
SHORT_TEST_HYDROGEN_PRODUCTION_ML_PER_INPUT_C = (
    charge_summary["hydrogen_volume_mL"].sum() / charge_summary["input_charge_C"].sum()
)
SHORT_TEST_HYDROGEN_COULOMB_EFFICIENCY = (
    SHORT_TEST_HYDROGEN_PRODUCTION_ML_PER_INPUT_C / FARADAY_THEORETICAL_H2_ML_PER_C
)

print(f"Short-test aggregate H2 production: {SHORT_TEST_HYDROGEN_PRODUCTION_ML_PER_INPUT_J:.4f} mL/J")
print(f"Short-test aggregate H2 production: {SHORT_TEST_HYDROGEN_PRODUCTION_ML_PER_INPUT_C:.4f} mL/C")
print(f"Short-test coulomb efficiency:       {100 * SHORT_TEST_HYDROGEN_COULOMB_EFFICIENCY:.1f} %")
print("A value above 100 percent means these small manual volume readings are noisy.")

plt.figure(figsize=(8, 4))
plt.scatter(charge_summary["input_energy_J"], charge_summary["hydrogen_volume_mL"])
x = np.linspace(0, charge_summary["input_energy_J"].max(), 100)
plt.plot(x, SHORT_TEST_HYDROGEN_PRODUCTION_ML_PER_INPUT_J * x, label="short-test aggregate fit")
plt.title("Hydrogen volume vs input energy")
plt.xlabel("Input energy [J]")
plt.ylabel("Measured hydrogen [mL]")
plt.grid(True)
plt.legend()
plt.show()


## 6. Use the full 15.6 mL calibration for the EMS value

The short tests are useful for checking trends, but the volume readings are small and therefore noisy.
For the final EMS production constant I use the larger calibration from the polarization file:

measured full hydrogen = 15.6 mL

hydrogen_per_input_J = 15.6 mL / input_energy_J

This gives a Faraday efficiency below 100 percent, which is more physically believable.


In [ ]:
polarization_file = PROJECT_ROOT / "data/PEM_test/current_sweep/PEM_polarization_characteristics.csv"
polarization = pd.read_csv(polarization_file)
polarization["timestamp"] = pd.to_datetime(polarization["timestamp"])
polarization["time_s"] = (polarization["timestamp"] - polarization["timestamp"].iloc[0]).dt.total_seconds()
polarization["pem_voltage_V"] = polarization["ina4_bus_V"] - 0.064
polarization["pem_current_A"] = 0.843 * (polarization["ina4_current_mA"] / 1000) + 0.001
polarization["pem_charge_power_W"] = polarization["pem_voltage_V"] * polarization["pem_current_A"]

polarization_charge = polarization[
    ((polarization["scenario"] == 3) | (polarization["pem_current_A"] > 0.01))
    & (polarization["pem_current_A"] > 0.05)
].copy()
polarization_charge["charge_time_s"] = (
    polarization_charge["time_s"] - polarization_charge["time_s"].iloc[0]
)

full_input_charge_C = np.trapezoid(
    polarization_charge["pem_current_A"],
    polarization_charge["charge_time_s"],
)
full_input_energy_J = np.trapezoid(
    polarization_charge["pem_charge_power_W"],
    polarization_charge["charge_time_s"],
)

MEASURED_FULL_HYDROGEN_CAPACITY_ML = 15.6
HYDROGEN_PRODUCTION_ML_PER_INPUT_J = MEASURED_FULL_HYDROGEN_CAPACITY_ML / full_input_energy_J
HYDROGEN_PRODUCTION_ML_PER_INPUT_C = MEASURED_FULL_HYDROGEN_CAPACITY_ML / full_input_charge_C
HYDROGEN_COULOMB_EFFICIENCY = HYDROGEN_PRODUCTION_ML_PER_INPUT_C / FARADAY_THEORETICAL_H2_ML_PER_C

print(f"Full calibration input charge: {full_input_charge_C:.2f} C")
print(f"Full calibration input energy: {full_input_energy_J:.2f} J")
print(f"EMS H2 production constant:    {HYDROGEN_PRODUCTION_ML_PER_INPUT_J:.4f} mL/J")
print(f"Faraday efficiency:            {100 * HYDROGEN_COULOMB_EFFICIENCY:.1f} %")


## 7. Values to use in the app


In [ ]:
MAXIMUM_ELECTROLYSIS_CURRENT_A = charge_summary["current_setpoint_A"].max()
MINIMUM_CHARGE_TIME_BEFORE_USEFUL_DISCHARGE_S = charge_summary["duration_setpoint_s"].min()

charge_parameters = pd.DataFrame(
    {
        "parameter": [
            "MEASURED_FULL_HYDROGEN_CAPACITY_ML",
            "HYDROGEN_PRODUCTION_ML_PER_INPUT_J",
            "HYDROGEN_PRODUCTION_ML_PER_INPUT_C",
            "HYDROGEN_COULOMB_EFFICIENCY",
            "SHORT_TEST_HYDROGEN_PRODUCTION_ML_PER_INPUT_J",
            "MAXIMUM_ELECTROLYSIS_CURRENT_A",
            "MINIMUM_CHARGE_TIME_BEFORE_USEFUL_DISCHARGE_S",
        ],
        "value": [
            MEASURED_FULL_HYDROGEN_CAPACITY_ML,
            HYDROGEN_PRODUCTION_ML_PER_INPUT_J,
            HYDROGEN_PRODUCTION_ML_PER_INPUT_C,
            HYDROGEN_COULOMB_EFFICIENCY,
            SHORT_TEST_HYDROGEN_PRODUCTION_ML_PER_INPUT_J,
            MAXIMUM_ELECTROLYSIS_CURRENT_A,
            MINIMUM_CHARGE_TIME_BEFORE_USEFUL_DISCHARGE_S,
        ],
        "unit": ["mL", "mL/J", "mL/C", "-", "mL/J", "A", "s"],
        "meaning": [
            "Measured full tank volume used for the demo",
            "Hydrogen produced per electrical input energy from full calibration",
            "Hydrogen produced per input charge from full calibration",
            "Full calibration efficiency relative to Faraday estimate",
            "Short-test value shown only as a noisy comparison",
            "Largest tested electrolysis current",
            "Smallest tested charge duration that produced measurable H2",
        ],
    }
)

display(charge_parameters)
